# KQD Error Characterisation — Fourier Box (Discrete Density of States)

This notebook characterises how errors grow in **Krylov Quantum Diagonalisation (KQD)**
when applied to a synthetic system whose spectrum is a **linearly-spaced density of states**
— the quantum analogue of a 1-D particle-in-a-box (Fourier box) with $N$ discrete
momentum modes.

### Physical setup

| Quantity | Definition |
|---|---|
| Spectrum | $E_k = k\,\Delta E$, $k=0,1,\ldots,N-1$, $\Delta E=1$ |
| Ground state $\lvert g\rangle$ | Equal-weight superposition of all $N$ eigenstates |
| Reference state $\lvert \psi_0\rangle$ | Configurable superposition; controls how much each eigenstate is sampled |
| KQD goal | Recover $E_0 = 0$ (lowest eigenvalue) from the Krylov subspace |

### Two error drivers investigated

1. **Initial-state distribution** — how the overlap amplitudes $\langle k\vert\psi_0\rangle$ are
   distributed across the spectrum affects how quickly the Krylov vectors resolve the ground state.
2. **Time-step choice** — $dt = s\,dt_{\text{opt}}$ with $s\in[0.1,\,1.5]$;
   too small → linearly-dependent Krylov vectors; too large → spectral wrap-around.

### Connection to the Fourier box

A particle in a 1-D box of length $L$ has eigenenergies
$E_n = \frac{\hbar^2\pi^2 n^2}{2mL^2}$.
With $N$ discrete Fourier points and a linear spacing approximation
(low-mode limit) this reduces exactly to the linear spectrum above.
The equal-superposition ground state mirrors a spatially flat initial wavefunction.


In [ ]:
import numpy as np
import scipy.linalg as la
import matplotlib
import matplotlib.pyplot as plt
import warnings, os

os.makedirs("output", exist_ok=True)
warnings.filterwarnings("ignore")

print("Imports OK")


## Step 1 — Build the Fourier-box Hamiltonian

The Hamiltonian is diagonal in the energy eigenbasis:

$$H = \sum_{k=0}^{N-1} E_k \lvert k\rangle\langle k\rvert, \qquad E_k = k\,\Delta E$$

We work directly in the eigenbasis, so $H$ is an $N\times N$ diagonal matrix.
The ground state is the **equal-weight superposition**

$$\lvert g\rangle = \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1}\lvert k\rangle$$

whose energy is

$$E_0^{\text{target}} = \langle g\vert H\vert g\rangle = \frac{N-1}{2}\Delta E$$

> **Note:** we label the *lowest eigenvalue* of $H$ as $E_{\min}=0$.  
> The KQD task is to recover $E_{\min}$ from a Krylov subspace seeded by $\lvert\psi_0\rangle$.


In [ ]:
def build_fourier_box(N: int, dE: float = 1.0):
    """
    Return (H, E_min, psi_ground) for an N-level linear spectrum.

    H          : (N,N) diagonal Hamiltonian  diag(0, dE, 2dE, ..., (N-1)dE)
    E_min      : float  lowest eigenvalue (= 0 by construction)
    psi_ground : (N,)   equal-weight superposition of ALL eigenstates
                        (This is NOT the ground eigenstate of H;
                         it is the target state we want KQD to find energy for.)
    """
    energies = np.arange(N, dtype=float) * dE          # 0, dE, 2dE, …
    H = np.diag(energies).astype(complex)
    E_min = energies[0]                                 # = 0
    psi_ground = np.ones(N, dtype=complex) / np.sqrt(N)  # equal superposition
    return H, E_min, psi_ground

# Quick sanity check
H6, E_min6, g6 = build_fourier_box(6)
print(f"N=6  eigenvalues : {np.diag(H6).real}")
print(f"     E_min       : {E_min6}")
print(f"     <g|H|g>     : {(g6.conj() @ H6 @ g6).real:.6f}  (expect {(6-1)/2})")
print(f"     ||g||       : {np.linalg.norm(g6):.6f}  (expect 1)")


## Step 2 — Reference State Distributions

The reference (initial) state $\lvert\psi_0\rangle$ is parameterised by its
overlap amplitudes with the $N$ eigenstates.  We study three families:

| Name | Amplitudes $c_k$ | Physical meaning |
|---|---|---|
| **Flat** | $c_k = 1/\sqrt{N}$ | Equal excitation of all modes — identical to the target ground state |
| **Gaussian** | $c_k \propto e^{-(k-\mu)^2/2\sigma^2}$ | Energy concentrated near mode $\mu$ |
| **Low-mode** | $c_k \propto e^{-\alpha k}$ | Exponential decay; heavily weights low-energy modes |

All states are normalised to unit norm before use.

The **overlap** $\lvert\langle g\vert\psi_0\rangle\rvert^2$ sets the minimum
Krylov dimension needed for convergence — small overlap means slow convergence
and larger errors at fixed dimension.


In [ ]:
def make_reference_state(N: int, kind: str = "flat", **kwargs) -> np.ndarray:
    """
    Build and normalise a reference state vector of length N.

    kind = 'flat'     : equal-weight superposition (identical to ground state)
    kind = 'gaussian' : Gaussian in mode index; kwargs: mu (default N/2), sigma (default N/4)
    kind = 'low_mode' : exponential decay;       kwargs: alpha (default 0.5)
    kind = 'custom'   : pass amplitudes=np.array([...]) directly
    """
    k = np.arange(N, dtype=float)
    if kind == "flat":
        c = np.ones(N, dtype=complex)
    elif kind == "gaussian":
        mu    = kwargs.get("mu",    N / 2)
        sigma = kwargs.get("sigma", N / 4)
        c = np.exp(-0.5 * ((k - mu) / sigma) ** 2).astype(complex)
    elif kind == "low_mode":
        alpha = kwargs.get("alpha", 0.5)
        c = np.exp(-alpha * k).astype(complex)
    elif kind == "custom":
        c = np.array(kwargs["amplitudes"], dtype=complex)
    else:
        raise ValueError(f"Unknown kind: {kind!r}")
    return c / np.linalg.norm(c)


# ── Demo ──────────────────────────────────────────────────────────────────────
N = 16
_, _, g = build_fourier_box(N)

states_demo = {
    "Flat"          : make_reference_state(N, "flat"),
    "Gaussian(μ=8)" : make_reference_state(N, "gaussian", mu=8, sigma=3),
    "Low-mode(α=0.5)": make_reference_state(N, "low_mode", alpha=0.5),
}

print(f"{'State':20s}  {'||psi||':>8s}  {'|<g|psi>|^2':>12s}")
print("-" * 45)
for name, psi in states_demo.items():
    ov = abs(g.conj() @ psi) ** 2
    print(f"{name:20s}  {np.linalg.norm(psi):8.6f}  {ov:12.6f}")


## Step 3 — KQD Engine (Exact Evolution)

The Krylov basis is built by repeated application of the time-evolution operator:

$$\lvert k\rangle = e^{-iH\,k\,dt}\lvert\psi_0\rangle, \quad k = 0,1,\ldots,d-1$$

Because $H$ is diagonal, $e^{-iHt}$ is trivially computed as
$\text{diag}(e^{-iE_0 t}, e^{-iE_1 t},\ldots)$ — no matrix exponentiation needed.

The optimal time step follows Epperly, Lin & Nakatsukasa (2022):

$$dt_{\text{opt}} = \frac{\pi}{\|H\|_2} = \frac{\pi}{E_{N-1}}$$

The overlap matrix $S_{jk} = \langle j\vert k\rangle$ and effective Hamiltonian
$\tilde{H}_{jk} = \langle j\vert H\vert k\rangle$ are assembled and the
regularised generalised eigenvalue problem $\tilde{H}\mathbf{c} = E\,S\mathbf{c}$
is solved.  Eigenvalues of $S$ below a threshold $\epsilon$ are projected out.


In [ ]:
def compute_dt_opt(H: np.ndarray) -> float:
    """dt_opt = pi / spectral_norm(H)."""
    return float(np.pi / np.linalg.norm(H, ord=2))


def build_krylov_states_diag(energies: np.ndarray, psi0: np.ndarray,
                              krylov_dim: int, dt: float) -> np.ndarray:
    """
    Build Krylov states for a diagonal Hamiltonian.
    Returns (krylov_dim, N) array; row k is exp(-i H k dt)|psi0>.
    """
    N   = len(energies)
    kst = np.zeros((krylov_dim, N), dtype=complex)
    kst[0] = psi0.copy()
    prop = np.exp(-1j * energies * dt)          # single-step propagator
    for j in range(1, krylov_dim):
        kst[j] = prop * kst[j - 1]             # O(N) per step
    return kst


def build_krylov_matrices(kst: np.ndarray, energies: np.ndarray):
    """Build S (overlap) and H_tilde (effective Hamiltonian) from Krylov states."""
    d = len(kst)
    S  = (kst.conj() @ kst.T)                  # d×d, O(d^2 N)
    Hv = energies[None, :] * kst               # H applied element-wise (diagonal H)
    Ht = (kst.conj() @ Hv.T)
    return S, Ht


def solve_gen_eig(Ht: np.ndarray, S: np.ndarray,
                  threshold: float = 1e-10):
    """Regularised generalised eigenvalue problem; returns (E_gs, eff_dim)."""
    import scipy.linalg as _la
    svals, svecs = _la.eigh(S)
    svecs = svecs.T                                  # rows are eigenvectors
    good  = [v for val, v in zip(svals, svecs) if val > threshold]
    if not good:
        return np.nan, 0
    P    = np.array(good)                            # (n_good, d) projection
    Hreg = P.conj() @ Ht @ P.T
    Sreg = P.conj() @ S  @ P.T
    E_gs = float(_la.eigh(Hreg, Sreg)[0][0].real)   # scipy handles generalised form
    return E_gs, len(good)


def run_kqd(energies: np.ndarray, psi0: np.ndarray,
            krylov_dim: int, dt: float,
            threshold: float = 1e-10) -> dict:
    """Full KQD run; returns dict with gs_energy and eff_dim."""
    kst      = build_krylov_states_diag(energies, psi0, krylov_dim, dt)
    S, Ht    = build_krylov_matrices(kst, energies)
    E_gs, ed = solve_gen_eig(Ht, S, threshold)
    return {"gs_energy": E_gs, "eff_dim": ed}


# ── Sanity check ───────────────────────────────────────────────────────────────
N = 16
H16, E_min16, g16 = build_fourier_box(N)
energies16 = np.diag(H16).real
psi0_flat  = make_reference_state(N, "flat")
dt_opt16   = compute_dt_opt(H16)

r = run_kqd(energies16, psi0_flat, krylov_dim=8, dt=dt_opt16)
print(f"KQD E_gs = {r['gs_energy']:.10f}   exact E_min = {E_min16:.10f}")
print(f"Error    = {abs(r['gs_energy'] - E_min16):.4e}")
print(f"dt_opt   = {dt_opt16:.6f}")


## Step 4 — Error Sweep: Initial-State Distribution vs. Time Step

We now sweep **jointly** over:

* **Reference state** — flat, Gaussian (narrow / wide), low-mode (fast / slow decay)
* **Time-step scale** $s \in [0.1,\,1.5]\,dt_{\text{opt}}$

and record the KQD error

$$\varepsilon = \lvert E_{\text{KQD}} - E_{\min}\rvert$$

at a fixed Krylov dimension $d$ for each combination.

This produces a **2-D error map** (state distribution × time step) that directly
shows how these two drivers interact.


In [ ]:
N          = 32          # number of Fourier / DOS levels
DIM        = 12          # fixed Krylov dimension for the sweep
DT_SCALES  = np.linspace(0.05, 1.5, 60)
THRESHOLD  = 1e-10

H32, E_min32, _ = build_fourier_box(N)
energies32      = np.diag(H32).real
dt_opt32        = compute_dt_opt(H32)

# ── Reference state families ──────────────────────────────────────────────────
ref_states = {
    "Flat"              : make_reference_state(N, "flat"),
    "Gaussian μ=N/2 σ=N/4" : make_reference_state(N, "gaussian", mu=N/2,  sigma=N/4),
    "Gaussian μ=N/2 σ=N/8" : make_reference_state(N, "gaussian", mu=N/2,  sigma=N/8),
    "Low-mode α=0.3"    : make_reference_state(N, "low_mode",  alpha=0.3),
    "Low-mode α=1.0"    : make_reference_state(N, "low_mode",  alpha=1.0),
}

# ── Compute overlap with equal-superposition ──────────────────────────────────
g32 = np.ones(N, dtype=complex) / np.sqrt(N)
overlaps = {k: abs(g32.conj() @ v)**2 for k, v in ref_states.items()}
print("Ground-state overlaps |<g|psi0>|^2:")
for k, ov in overlaps.items():
    print(f"  {k:35s}  {ov:.6f}")

# ── Sweep ─────────────────────────────────────────────────────────────────────
sweep_errors = {}           # key: ref state name → array of errors over dt scales
for name, psi0 in ref_states.items():
    errs = []
    for s in DT_SCALES:
        dt  = s * dt_opt32
        res = run_kqd(energies32, psi0, DIM, dt, THRESHOLD)
        err = abs(res["gs_energy"] - E_min32) if not np.isnan(res["gs_energy"]) else np.nan
        errs.append(err)
    sweep_errors[name] = np.array(errs)

print("\nSweep complete.")


## Step 5 — Convergence vs. Krylov Dimension at Fixed $dt$

For each reference state we plot the KQD error as a function of Krylov dimension
$d = 1,\ldots,d_{\max}$ at the optimal time step $dt_{\text{opt}}$.

This isolates the **distributional** error driver: how quickly each initial-state
distribution converges to the ground state as the subspace grows.


In [ ]:
KRYLOV_DIMS = list(range(1, 20))
DT_FIXED    = dt_opt32            # fixed at dt_opt for this sweep

conv_errors = {}
for name, psi0 in ref_states.items():
    errs = []
    for d in KRYLOV_DIMS:
        res = run_kqd(energies32, psi0, d, DT_FIXED, THRESHOLD)
        err = abs(res["gs_energy"] - E_min32) if not np.isnan(res["gs_energy"]) else np.nan
        errs.append(err)
    conv_errors[name] = np.array(errs)

print("Convergence sweep complete.")


## Step 6 — Plots

Three panels:

1. **Error vs. $dt/dt_{\text{opt}}$** (log scale) — shows the sweet-spot and how
   initial-state distribution shifts it.
2. **Error vs. Krylov dimension** (log scale, $dt=dt_{\text{opt}}$) — shows
   convergence rate per distribution.
3. **2-D heat map** — error as a joint function of ($dt$ scale, reference state),
   colour-coded on a log scale.


In [ ]:
import matplotlib.ticker as ticker
from matplotlib.colors import LogNorm

CHEM_ACC = 1.6e-3   # chemical accuracy reference line
COLORS   = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Panel 1: Error vs dt scale ────────────────────────────────────────────────
ax = axes[0]
for i, (name, errs) in enumerate(sweep_errors.items()):
    ax.semilogy(DT_SCALES, errs, label=name, lw=1.8, color=COLORS[i % len(COLORS)])
ax.axhline(CHEM_ACC, color="gray", ls="--", lw=1.2, label="Chem. acc. (1.6 mHa)")
ax.axvline(1.0,      color="black", ls=":", lw=1.0, label="$dt_{\\mathrm{opt}}$")
ax.set_xlabel("$dt$ / $dt_{\\mathrm{opt}}$")
ax.set_ylabel("$|E_{\\mathrm{KQD}} - E_{\\min}|$")
ax.set_title(f"Error vs time-step (d={DIM}, N={N})")
ax.legend(fontsize=7)
ax.grid(True, which="both", alpha=0.3)

# ── Panel 2: Error vs Krylov dimension ───────────────────────────────────────
ax = axes[1]
for i, (name, errs) in enumerate(conv_errors.items()):
    ax.semilogy(KRYLOV_DIMS, errs, "o-", label=name, lw=1.8,
                color=COLORS[i % len(COLORS)], ms=4)
ax.axhline(CHEM_ACC, color="gray", ls="--", lw=1.2, label="Chem. acc.")
ax.set_xlabel("Krylov dimension $d$")
ax.set_ylabel("$|E_{\\mathrm{KQD}} - E_{\\min}|$")
ax.set_title(f"Convergence vs Krylov dim ($dt=dt_{{\\mathrm{{opt}}}}$, N={N})")
ax.legend(fontsize=7)
ax.grid(True, which="both", alpha=0.3)

# ── Panel 3: 2-D heat map ────────────────────────────────────────────────────
ax    = axes[2]
names = list(sweep_errors.keys())
mat   = np.array([sweep_errors[n] for n in names])   # (n_states, n_dt)
# Replace NaN / zero for log colour scale
mat_plot = np.where(np.isnan(mat) | (mat <= 0), 1e-16, mat)
im = ax.imshow(mat_plot, aspect="auto", origin="upper",
               norm=LogNorm(vmin=1e-14, vmax=mat_plot.max()),
               cmap="plasma",
               extent=[DT_SCALES[0], DT_SCALES[-1], len(names)-0.5, -0.5])
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=7)
ax.set_xlabel("$dt$ / $dt_{\\mathrm{opt}}$")
ax.set_title("Log-error heat map (d={}, N={})".format(DIM, N))
plt.colorbar(im, ax=ax, label="$|E_{\\mathrm{KQD}} - E_{\\min}|$")
ax.axvline(1.0, color="white", ls=":", lw=1.2)

plt.tight_layout()
plt.savefig("output/kqd_error_characterisation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved → output/kqd_error_characterisation.png")


## Step 7 — Fine $dt$ Sensitivity: Error Growth Rate

To quantify how fast errors *grow* away from $dt_{\text{opt}}$, we fit a
power-law slope to the error vs. $|s - 1|$ curve on each side of the optimum.

This characterises the **asymmetry** of the error landscape:
- Left of $dt_{\text{opt}}$ (small $dt$): linear-dependence regime → error often grows steeply
- Right of $dt_{\text{opt}}$ (large $dt$): wrap-around regime → error can grow more gradually


In [ ]:
from scipy.stats import linregress

# Use flat reference state for the slope analysis
psi0_flat = ref_states["Flat"]
DT_FINE   = np.linspace(0.05, 1.5, 200)
DIMS_FINE = [4, 8, 12, 16]

fig, ax = plt.subplots(figsize=(9, 5))
for d in DIMS_FINE:
    errs = []
    for s in DT_FINE:
        res = run_kqd(energies32, psi0_flat, d, s * dt_opt32, THRESHOLD)
        err = abs(res["gs_energy"] - E_min32) if not np.isnan(res["gs_energy"]) else np.nan
        errs.append(err)
    errs = np.array(errs)
    ax.semilogy(DT_FINE, errs, lw=1.6, label=f"d={d}")

ax.axhline(CHEM_ACC, color="gray",  ls="--", lw=1.2, label="Chem. acc.")
ax.axvline(1.0,      color="black", ls=":",  lw=1.0, label="$dt_{\\mathrm{opt}}$")
ax.set_xlabel("$dt$ / $dt_{\\mathrm{opt}}$")
ax.set_ylabel("$|E_{\\mathrm{KQD}} - E_{\\min}|$")
ax.set_title(f"Error vs dt scale — flat reference, N={N}")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("output/kqd_dt_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved → output/kqd_dt_sensitivity.png")


## Step 8 — Interpretation Guide

| Observation | Cause | Mitigation |
|---|---|---|
| Error **plateaus** at large $d$ for Gaussian/low-mode states | Overlap $\lvert\langle g\vert\psi_0\rangle\rvert^2$ is small; subspace can't fully resolve $E_{\min}$ | Use a flatter (higher-overlap) reference state |
| Error **spikes** at small $dt$ | Consecutive Krylov vectors are nearly parallel ($S$ becomes rank-deficient) | Increase $dt$ or use larger regularisation threshold $\epsilon$ |
| Error **spikes** at large $dt$ | Spectral wrap-around: $e^{-iE_{N-1}dt}$ completes full rotations | Stay below $dt_{\text{opt}}$ |
| Narrow Gaussian converges **slower** than wide Gaussian | Narrower overlap → fewer eigenstate components resolved per Krylov step | Widen the reference distribution |
| Low-mode state converges **fast at low $d$** but stalls | Dominant low-energy overlap resolved early; high modes need more vectors | Add more Krylov steps |

### Key takeaway

The **two main error drivers** are:

1. $\lvert\langle g\vert\psi_0\rangle\rvert^2$ — the ground-state overlap of the
   reference state.  This sets the *floor* of achievable error at any fixed Krylov
   dimension.
2. The **time-step choice** $dt/dt_{\text{opt}}$.  The error landscape is
   asymmetric: the small-$dt$ cliff is sharper than the large-$dt$ slope, because
   linear dependence in $S$ is catastrophic whereas mild spectral wrap-around is
   partially corrected by the regulariser.


## References

1. Epperly, Lin & Nakatsukasa, *A theory of quantum subspace diagonalization*, SIAM J. Matrix Anal. Appl. (2022)
2. Yu et al., *Quantum-centric algorithm for sample-based Krylov diagonalization*, arXiv:2501.09702 (2025)
3. Hatano & Suzuki, *Finding Exponential Product Formulas of Higher Orders* (2005)
